# Filter and Group Trial Results: Time Series + Scalars

This cookbook shows how to:
- filter patients with `get_filtered_patients`
- download time series with `get_timeseries_as_dataframe`
- download grouped scalars with `get_trial_scalars_with_filter_and_groups_as_dataframe`

Linked resource: [Jinko](https://jinko.ai/tr-pPqG-BqG2).

In [ ]:
# Jinko specifics imports & initialization
from jinko import JinkoClient

client = JinkoClient()
client.auth_check()

In [ ]:
# Cookbook specifics imports
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

# Cookbook parameters
# @param {"name":"trialId", "type": "string"}
# Fill the short Id of your Trial (ex: tr-EKRx-3HRt)
trialId = "tr-pPqG-BqG2"

# Optional: limit how many time series to download in this demo
max_time_series = 5

# Filter parameters (for patients_matching_filters)
arm_filter = "iv-1-30"
filter_descriptor_id = "Lymph.Drug.max"
filter_operator = "Lte"
filter_value = 5e-3

# Group parameters (for scalars_per_population)
group_descriptor_id = "Lymph.tmin"
group_buckets = 2

# Scalar descriptors to download
scalar_ids = ["Blood.Drug-at-P1W3D"]

# Scalar descriptors to download along with their arms
scalar_ids_on_specific_arms = {"Blood.Drug-at-P1W3D": ["iv-1-10"]}

## 1) Load the trial snapshot

In [ ]:
if not trialId:
    raise Exception("Please specify a Trial Id")

# Load the latest revision and make sure it actually completed.
# There is no high-level "latest completed" lookup in the installed SDK
# (jinko-sdk 1.3.1), so this is the documented API-gap workaround: fetch
# the latest revision and check its status explicitly.
try:
    trial = client.get_trial(trialId)
    status = trial.status()
    if status.get("status") != "completed":
        raise Exception(
            f"Trial {trialId} latest revision is not completed (status: {status.get('status')})"
        )
except Exception as e:
    print(f"Error fetching completed trial version: {e}")
    raise

print(
    f"Using completed trial: {trialId} (coreItemId: {trial.core_id}, snapshotId: {trial.snapshot_id})")

## 2) Display a results summary

In [ ]:
summary = trial.results.summary()

arm_names = summary["arms"]

# Store the list of scenario descriptors
scenario_descriptors = [
    scalar["id"]
    for scalar in (summary["scalars"] + summary["categoricals"])
    if "ScenarioOverride" in scalar["type"]["labels"]
]
print("List of scenario overrides:\n", scenario_descriptors)

## 3) Define filters and groups

In [ ]:
# filter type definition: https://doc.jinko.ai/api/#/schemas/Filter
filter_on_scalar = [
    {
        "contents": [
            {
                "arm": arm_filter,
                "descriptorId": filter_descriptor_id,
                "operator": filter_operator,
                "value": filter_value,
            }
        ],
        "tag": "DescriptorFilter",
    }
]

# group type definition: https://doc.jinko.ai/api/#/schemas/GroupBy
group = [
    {
        "contents": {
            "arm": "crossArms",
            "descriptorId": group_descriptor_id,
            "mode": "Buckets",
            "parameters": group_buckets,
        },
        "tag": "Scalar",
    }
]

## 4) Get filtered patients list

In [ ]:
patient_ids_to_keep = trial.results.patients_matching_filters(
    filter_tokens=filter_on_scalar
)
print(f"Filtered patients: {len(patient_ids_to_keep)}")

In [ ]:
patient_ids_to_keep

## 5) Download time series

In [ ]:
# Retrieve time series ids
available_time_series = trial.output_ids()
print("Available time series:")
print(available_time_series, "")

ids_for_time_series = [x["id"]
                       for x in available_time_series][:max_time_series]
print("Using time series ids:", ids_for_time_series)

In [ ]:
# `trial.results.timeseries(...)` has no server-side patient filter (SDK gap
# vs the legacy `get_timeseries_as_dataframe(..., patient_ids_to_keep)`), so we
# download the full timeseries and filter to the matching patients client-side.
df_time_series = trial.results.timeseries(
    {ts_id: arm_names for ts_id in ids_for_time_series}
).to_dataframe()
df_time_series = df_time_series[df_time_series["Patient Id"].isin(
    patient_ids_to_keep)]

display(df_time_series)

## 6) Download grouped scalar results

In [ ]:
# `trial.results.scalars_per_population(...)` returns the raw nested
# per_population endpoint payload rather than a dataframe (SDK gap vs the
# legacy `get_trial_scalars_with_filter_and_groups_as_dataframe(...)`), so we
# flatten it ourselves. Verified live: each entry in `group_key` carries a
# `contents` dict describing the actual bucket (e.g. `descriptorId` plus a
# `parameters.lowBound`/`highBound` span for `mode: "Buckets"`/"Span") — using
# only `tag` collapses every bucket into an indistinguishable "Scalar" label,
# so we build a human-readable group label from `contents` instead.
def _describe_group_token(token):
    contents = token.get("contents", {})
    descriptor_id = contents.get("descriptorId")
    params = contents.get("parameters")
    if isinstance(params, dict) and "lowBound" in params:
        return f"{descriptor_id} in [{params['lowBound']:.3g}, {params['highBound']:.3g})"
    return f"{descriptor_id}={params}" if descriptor_id else token.get("tag")


def scalars_per_population_as_dataframe(response):
    rows = []
    for group_key, arms in response.get("scalars", []):
        group_label = tuple(_describe_group_token(g) for g in group_key)
        for arm, scalars in arms.items():
            for scalar_id, payload in scalars.items():
                if not payload:
                    continue
                unit = payload.get("unit")
                for value in payload.get("values", []):
                    rows.append(
                        {
                            "group": group_label,
                            "armId": arm,
                            "scalarId": scalar_id,
                            "value": value,
                            "unit": unit,
                        }
                    )
    return pd.DataFrame(rows)


response = trial.results.scalars_per_population(
    scalars=scalar_ids,
    filter_tokens=filter_on_scalar,
    group_tokens=group,
)
df_scalars = scalars_per_population_as_dataframe(response)

display(df_scalars)

## 7) Download grouped scalar results but only on one arm

In [ ]:
response = trial.results.scalars_per_population(
    scalars=scalar_ids_on_specific_arms,
    filter_tokens=filter_on_scalar,
    group_tokens=group,
)
df_scalars = scalars_per_population_as_dataframe(response)

display(df_scalars)